In [ ]:
!pip install -q seqeval

In [ ]:
import os
import json
import random
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from safetensors.torch import load_file as load_safetensors
from seqeval.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    PretrainedConfig,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")

SEED = 42
IGNORE_INDEX = -100
DATASET_NAME = "phucdev/PhoNER_COVID19"


def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:
# Colab Drive path chua cac folder checkpoint: CNN, DistilBERT, ELECTRA, ...
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Skip drive.mount:", exc)

DRIVE_ROOT = Path("/content/drive/MyDrive/Mini Project NLP")
RESULTS_DIR = DRIVE_ROOT / "experimental_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)


Mounted at /content/drive
DRIVE_ROOT: /content/drive/MyDrive/Mini Project NLP
RESULTS_DIR: /content/drive/MyDrive/Mini Project NLP/experimental_results


In [ ]:
def normalize_batch(examples):
    return {
        "words": [
            [unicodedata.normalize("NFC", word) for word in words]
            for words in examples["words"]
        ]
    }


def load_clean_dataset(variant):
    dataset = load_dataset(DATASET_NAME, variant)
    return dataset.map(normalize_batch, batched=True, num_proc=2)


clean_datasets = {
    "word": load_clean_dataset("word"),
    "syllable": load_clean_dataset("syllable"),
}

label_maps = {}
for variant, dataset in clean_datasets.items():
    unique_tags = sorted({tag for tags in dataset["train"]["tags"] for tag in tags})
    label2idx = {tag: idx for idx, tag in enumerate(unique_tags)}
    idx2label = {idx: tag for tag, idx in label2idx.items()}
    label_maps[variant] = {
        "unique_tags": unique_tags,
        "label2idx": label2idx,
        "idx2label": idx2label,
    }
    print(f"{variant}: {len(unique_tags)} labels")
    print(label2idx)


def get_label_map(variant):
    return label_maps[variant]["label2idx"], label_maps[variant]["idx2label"]


def num_labels(variant):
    return len(label_maps[variant]["unique_tags"])

# Backward-compatible defaults for custom word-level models.
unique_tags = label_maps["word"]["unique_tags"]
label2idx = label_maps["word"]["label2idx"]
idx2label = label_maps["word"]["idx2label"]


README.md:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

word/train-00000-of-00001.parquet:   0%|          | 0.00/366k [00:00<?, ?B/s]

word/validation-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

word/test-00000-of-00001.parquet:   0%|          | 0.00/247k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5027 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/5027 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/3000 [00:00<?, ? examples/s]

word: 20 labels
{'B-AGE': 0, 'B-DATE': 1, 'B-GENDER': 2, 'B-JOB': 3, 'B-LOCATION': 4, 'B-NAME': 5, 'B-ORGANIZATION': 6, 'B-PATIENT_ID': 7, 'B-SYMPTOM_AND_DISEASE': 8, 'B-TRANSPORTATION': 9, 'I-AGE': 10, 'I-DATE': 11, 'I-JOB': 12, 'I-LOCATION': 13, 'I-NAME': 14, 'I-ORGANIZATION': 15, 'I-PATIENT_ID': 16, 'I-SYMPTOM_AND_DISEASE': 17, 'I-TRANSPORTATION': 18, 'O': 19}
syllable: 21 labels
{'B-AGE': 0, 'B-DATE': 1, 'B-GENDER': 2, 'B-JOB': 3, 'B-LOCATION': 4, 'B-NAME': 5, 'B-ORGANIZATION': 6, 'B-PATIENT_ID': 7, 'B-SYMPTOM_AND_DISEASE': 8, 'B-TRANSPORTATION': 9, 'I-AGE': 10, 'I-DATE': 11, 'I-GENDER': 12, 'I-JOB': 13, 'I-LOCATION': 14, 'I-NAME': 15, 'I-ORGANIZATION': 16, 'I-PATIENT_ID': 17, 'I-SYMPTOM_AND_DISEASE': 18, 'I-TRANSPORTATION': 19, 'O': 20}


In [ ]:
TOKENIZER_NAMES = {
    "phobert": "vinai/phobert-base-v2",
    "distilbert": "distilbert/distilbert-base-multilingual-cased",
    "electra": "NlpHUST/electra-base-vn",
    "videberta": "manhtt-079/vipubmed-deberta-base",
    "xlm-roberta": "FacebookAI/xlm-roberta-base",
}

tokenizer_cache = {}
tokenized_cache = {}


def get_tokenizer(tokenizer_key):
    if tokenizer_key not in tokenizer_cache:
        tokenizer_cache[tokenizer_key] = AutoTokenizer.from_pretrained(
            TOKENIZER_NAMES[tokenizer_key],
            use_fast=True,
        )
    return tokenizer_cache[tokenizer_key]


def tokenize_phobert_like(examples, tokenizer, label2idx, max_length=None):
    all_input_ids = []
    all_labels = []
    all_attention_masks = []

    cls_token_id = tokenizer.cls_token_id
    sep_token_id = tokenizer.sep_token_id
    unk_token_id = tokenizer.unk_token_id

    for words, tags in zip(examples["words"], examples["tags"]):
        body_input_ids = []
        body_label_ids = []

        for word, tag in zip(words, tags):
            word_piece_ids = tokenizer.encode(word, add_special_tokens=False)
            if not word_piece_ids:
                word_piece_ids = [unk_token_id]

            body_input_ids.extend(word_piece_ids)
            body_label_ids.append(label2idx[tag])
            if len(word_piece_ids) > 1:
                body_label_ids.extend([IGNORE_INDEX] * (len(word_piece_ids) - 1))

        input_ids = [cls_token_id] + body_input_ids + [sep_token_id]
        labels = [IGNORE_INDEX] + body_label_ids + [IGNORE_INDEX]

        if max_length is not None and len(input_ids) > max_length:
            input_ids = input_ids[:max_length]
            labels = labels[:max_length]

        attention_mask = [1] * len(input_ids)
        all_input_ids.append(input_ids)
        all_labels.append(labels)
        all_attention_masks.append(attention_mask)

    return {
        "input_ids": all_input_ids,
        "labels": all_labels,
        "attention_mask": all_attention_masks,
    }


def tokenize_fast_word_ids(examples, tokenizer, label2idx, max_length=None):
    tokenized_inputs = tokenizer(
        examples["words"],
        is_split_into_words=True,
        truncation=True,
        padding=False,
        max_length=max_length,
    )

    all_labels = []
    for batch_index, tags in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)
        label_ids = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(IGNORE_INDEX)
            elif word_id != previous_word_id:
                label_ids.append(label2idx[tags[word_id]])
            else:
                label_ids.append(IGNORE_INDEX)
            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


def get_tokenized_dataset(variant, tokenizer_key, strategy, max_length=None):
    cache_key = (variant, tokenizer_key, strategy, max_length)
    if cache_key in tokenized_cache:
        return tokenized_cache[cache_key]

    tokenizer = get_tokenizer(tokenizer_key)
    dataset = clean_datasets[variant]
    variant_label2idx = label_maps[variant]["label2idx"]

    if strategy == "phobert_manual":
        tokenize_fn = lambda examples: tokenize_phobert_like(
            examples,
            tokenizer,
            variant_label2idx,
            max_length=max_length,
        )
    elif strategy == "fast_word_ids":
        tokenize_fn = lambda examples: tokenize_fast_word_ids(
            examples,
            tokenizer,
            variant_label2idx,
            max_length=max_length,
        )
    else:
        raise ValueError(f"Unknown tokenization strategy: {strategy}")

    tokenized = dataset.map(
        tokenize_fn,
        batched=True,
        num_proc=2,
        remove_columns=["words", "tags"],
    )
    tokenized_cache[cache_key] = tokenized
    return tokenized


In [ ]:
class CNN1DResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size, dropout=0.1):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.norm1(out.transpose(1, 2)).transpose(1, 2)
        out = self.act(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.norm2(out.transpose(1, 2)).transpose(1, 2)
        return self.act(out + residual)


class CNN_SwiGLU(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.w1 = nn.Linear(in_features, out_features)
        self.w2 = nn.Linear(in_features, out_features)

    def forward(self, x):
        return F.silu(self.w1(x)) * self.w2(x)


class MultiScaleCNN1DModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 pad_idx=IGNORE_INDEX, num_blocks=2, kernel_sizes=(3, 5, 7), dropout=0.1):
        super().__init__()
        self.config = PretrainedConfig(
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            hidden_dim=hidden_dim,
            output_dim=output_dim,
            pad_idx=pad_idx,
            num_blocks=num_blocks,
            kernel_sizes=kernel_sizes,
            dropout=dropout,
        )
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embed_proj = nn.Linear(embedding_dim, hidden_dim)
        self.embed_norm = nn.LayerNorm(hidden_dim)
        self.embed_dropout = nn.Dropout(dropout)
        self.branches = nn.ModuleList([
            nn.Sequential(*[CNN1DResidualBlock(hidden_dim, ks, dropout) for _ in range(num_blocks)])
            for ks in kernel_sizes
        ])
        self.swiglu = CNN_SwiGLU(hidden_dim * len(kernel_sizes), hidden_dim)
        self.fusion_norm = nn.LayerNorm(hidden_dim)
        self.fusion_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, output_dim)
        self.pad_idx = pad_idx
        self.output_dim = output_dim

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        x = self.embedding(input_ids)
        x = F.gelu(self.embed_norm(self.embed_proj(x)))
        x = self.embed_dropout(x).transpose(1, 2)
        x = torch.cat([branch(x) for branch in self.branches], dim=1).transpose(1, 2)
        x = self.swiglu(x)
        x = self.fusion_norm(x)
        x = self.fusion_dropout(x)
        logits = self.classifier(F.gelu(x))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, self.output_dim),
                labels.reshape(-1),
                ignore_index=self.pad_idx,
            )
        return {"loss": loss, "logits": logits}


class ImprovedRNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, ignore_index,
                 pad_token_id, num_layers=1, dropout=0.1, bidirectional=True, nonlinearity="tanh"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_token_id)
        self.embedding_dropout = nn.Dropout(dropout)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
            nonlinearity=nonlinearity,
        )
        rnn_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.layer_norm = nn.LayerNorm(rnn_output_dim)
        self.output_dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(rnn_output_dim, output_dim)
        self.ignore_index = ignore_index
        self.output_dim = output_dim

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        x = self.embedding_dropout(self.embedding(input_ids))
        if attention_mask is not None:
            lengths = attention_mask.sum(dim=1).cpu().clamp(min=1)
            packed_x = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            packed_out, _ = self.rnn(packed_x)
            x, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=input_ids.size(1))
        else:
            x, _ = self.rnn(x)
        logits = self.fc(self.output_dropout(self.layer_norm(x)))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.output_dim), labels.reshape(-1), ignore_index=self.ignore_index)
        return {"loss": loss, "logits": logits}


class LSTM_SwiGLU(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.w = nn.Linear(input_dim, output_dim)
        self.v = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return F.silu(self.w(x)) * self.v(x)


class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, ignore_index,
                 num_layers=2, dropout=0.2, bidirectional=True, num_heads=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=ignore_index)
        self.spatial_dropout = nn.Dropout1d(dropout)
        self.norm_emb = nn.LayerNorm(embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional,
        )
        self.lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.attention = nn.MultiheadAttention(
            embed_dim=self.lstm_output_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm_attn = nn.LayerNorm(self.lstm_output_dim)
        intermediate_dim = int(self.lstm_output_dim * 2.0)
        self.classifier = nn.Sequential(
            LSTM_SwiGLU(self.lstm_output_dim, intermediate_dim),
            nn.Dropout(dropout),
            nn.Linear(intermediate_dim, output_dim),
        )
        self.ignore_index = ignore_index
        self.output_dim = output_dim

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        x = self.embedding(input_ids).transpose(1, 2)
        x = self.spatial_dropout(x).transpose(1, 2)
        x = self.norm_emb(x)
        if attention_mask is not None:
            lengths = attention_mask.sum(dim=1).cpu().clamp(min=1)
            packed_x = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            packed_out, _ = self.lstm(packed_x)
            lstm_out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))
        else:
            lstm_out, _ = self.lstm(x)
        attn_out, _ = self.attention(query=lstm_out, key=lstm_out, value=lstm_out)
        context = self.norm_attn(lstm_out + attn_out)
        logits = self.classifier(context)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.output_dim), labels.reshape(-1), ignore_index=self.ignore_index)
        return {"loss": loss, "logits": logits}


class GRU_SwiGLUBlock(nn.Module):
    def __init__(self, input_dim, ffn_dim, dropout=0.1):
        super().__init__()
        self.fc = nn.Linear(input_dim, ffn_dim * 2)
        self.proj = nn.Linear(ffn_dim, input_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(input_dim)

    def forward(self, x):
        residual = x
        value, gate = self.fc(x).chunk(2, dim=-1)
        x = value * F.silu(gate)
        return self.norm(self.dropout(self.proj(x)) + residual)


class BiGRUSwiGLUModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 num_layers=1, dropout=0.1, ffn_dim=512, ignore_index=IGNORE_INDEX, bidirectional=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        gru_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.swiglu = GRU_SwiGLUBlock(gru_output_dim, ffn_dim, dropout)
        self.classifier_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(gru_output_dim, output_dim)
        self.ignore_index = ignore_index
        self.output_dim = output_dim

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        embedded = self.embedding_dropout(self.embedding(input_ids))
        gru_output, _ = self.gru(embedded)
        logits = self.classifier(self.classifier_dropout(self.swiglu(gru_output)))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.output_dim), labels.reshape(-1), ignore_index=self.ignore_index)
        return {"loss": loss, "logits": logits}


class MLPTokenClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, hidden_dim=256, output_dim=None,
                 num_layers=2, dropout=0.1, ignore_index=IGNORE_INDEX, pad_token_id=None):
        super().__init__()
        if output_dim is None:
            output_dim = len(unique_tags)
        padding_idx = pad_token_id if isinstance(pad_token_id, int) and pad_token_id >= 0 else None
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        layers = []
        in_dim = embedding_dim
        for _ in range(max(1, int(num_layers))):
            layers.extend([nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.LayerNorm(hidden_dim)])
            in_dim = hidden_dim
        self.mlp = nn.Sequential(*layers)
        self.classifier = nn.Linear(hidden_dim, output_dim)
        self.ignore_index = ignore_index
        self.output_dim = output_dim

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        logits = self.classifier(self.mlp(self.embedding(input_ids)))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.output_dim), labels.reshape(-1), ignore_index=self.ignore_index)
        return {"loss": loss, "logits": logits}


In [ ]:
MODEL_SPECS = [
    {
        "name": "CNN",
        "type": "custom",
        "variant": "word",
        "tokenizer_key": "phobert",
        "strategy": "phobert_manual",
        "max_length": 72,
        "folders": ["CNN", "cnn1d_multi_scale", "cnn1d_ner_final", "cnn1d_ner"],
        "batch_size": 256,
        "builder": "cnn",
        "defaults": {
            "vocab_size": 64000,
            "embedding_dim": 256,
            "hidden_dim": 256,
            "output_dim": num_labels("word"),
            "pad_idx": IGNORE_INDEX,
            "num_blocks": 2,
            "kernel_sizes": (3, 5, 7),
            "dropout": 0.15,
        },
    },
    {
        "name": "RNN",
        "type": "custom",
        "variant": "word",
        "tokenizer_key": "phobert",
        "strategy": "phobert_manual",
        "max_length": None,
        "folders": ["RNN", "improved_rnn", "improved_rnn_ner"],
        "batch_size": 256,
        "builder": "rnn",
        "defaults": {
            "vocab_size": 64000,
            "embedding_dim": 256,
            "hidden_dim": 256,
            "output_dim": num_labels("word"),
            "ignore_index": IGNORE_INDEX,
            "pad_token_id": get_tokenizer("phobert").pad_token_id,
            "num_layers": 1,
            "dropout": 0.1,
            "bidirectional": True,
            "nonlinearity": "tanh",
        },
    },
    {
        "name": "LSTM",
        "type": "custom",
        "variant": "word",
        "tokenizer_key": "phobert",
        "strategy": "phobert_manual",
        "max_length": None,
        "folders": ["LSTM", "lstm_ner_final", "lstm_ner"],
        "batch_size": 256,
        "builder": "lstm",
        "defaults": {
            "vocab_size": 64000,
            "embedding_dim": 256,
            "hidden_dim": 256,
            "output_dim": num_labels("word"),
            "ignore_index": IGNORE_INDEX,
            "num_layers": 2,
            "dropout": 0.1,
            "bidirectional": True,
            "num_heads": 4,
        },
    },
    {
        "name": "GRU",
        "type": "custom",
        "variant": "word",
        "tokenizer_key": "phobert",
        "strategy": "phobert_manual",
        "max_length": None,
        "folders": ["GRU", "bigru_swiglu_ner", "bigru_swiglu_checkpoints"],
        "batch_size": 256,
        "builder": "gru",
        "defaults": {
            "vocab_size": 64000,
            "embedding_dim": 256,
            "hidden_dim": 256,
            "output_dim": num_labels("word"),
            "num_layers": 1,
            "dropout": 0.1,
            "ffn_dim": 512,
            "ignore_index": IGNORE_INDEX,
            "bidirectional": True,
        },
    },
    {
        "name": "PhoBERT",
        "type": "transformer",
        "variant": "word",
        "tokenizer_key": "phobert",
        "strategy": "phobert_manual",
        "max_length": None,
        "base_model": "vinai/phobert-base-v2",
        "folders": ["PhoBERT", "phobert_ner_final", "phobert_ner"],
        "batch_size": 64,
    },
    {
        "name": "ViDeBERTa",
        "type": "transformer",
        "variant": "word",
        "tokenizer_key": "videberta",
        "strategy": "fast_word_ids",
        "max_length": None,
        "base_model": "manhtt-079/vipubmed-deberta-base",
        "folders": ["ViDeBERTa", "vidiberta_ner_final", "videberta_ner", "videbearta_ner"],
        "batch_size": 32,
    },
    {
        "name": "DistilBERT",
        "type": "transformer",
        "variant": "syllable",
        "tokenizer_key": "distilbert",
        "strategy": "fast_word_ids",
        "max_length": None,
        "base_model": "distilbert/distilbert-base-multilingual-cased",
        "folders": ["DistilBERT", "distilbert_ner_final", "distilbert_ner"],
        "batch_size": 64,
    },
    {
        "name": "ELECTRA",
        "type": "transformer",
        "variant": "syllable",
        "tokenizer_key": "electra",
        "strategy": "fast_word_ids",
        "max_length": None,
        "base_model": "NlpHUST/electra-base-vn",
        "folders": ["ELECTRA", "electra_ner_final", "electra_ner"],
        "batch_size": 64,
    },
    {
        "name": "XLM-RoBERTa",
        "type": "transformer",
        "variant": "syllable",
        "tokenizer_key": "xlm-roberta",
        "strategy": "fast_word_ids",
        "max_length": None,
        "base_model": "FacebookAI/xlm-roberta-base",
        "folders": ["XLM-RoBERTa", "XLM_RoBERTa", "xlm_roberta_ner_final", "xlm_roberta_ner"],
        "batch_size": 64,
    },
]


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [ ]:
def has_weight_file(path):
    path = Path(path)
    return any((path / name).exists() for name in ["model.safetensors", "pytorch_model.bin", "model.bin"])


def checkpoint_sort_key(path):
    name = Path(path).name
    if name.startswith("checkpoint-"):
        try:
            return int(name.split("-")[-1])
        except ValueError:
            pass
    return int(Path(path).stat().st_mtime)


def find_checkpoint_dir(spec):
    candidates = []
    for folder_name in spec["folders"]:
        root = DRIVE_ROOT / folder_name
        if not root.exists():
            continue
        if has_weight_file(root):
            candidates.append(root)
        for child in root.glob("checkpoint-*"):
            if child.is_dir() and has_weight_file(child):
                candidates.append(child)
        for child in root.iterdir() if root.is_dir() else []:
            if child.is_dir() and has_weight_file(child):
                candidates.append(child)

    if not candidates:
        return None
    return sorted(set(candidates), key=checkpoint_sort_key, reverse=True)[0]


def read_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_custom_config(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    for filename in ["model_config.json", "config.json"]:
        cfg = read_json_if_exists(checkpoint_dir / filename)
        if cfg:
            return cfg
    return {}


def load_state_dict_from_dir(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    if (checkpoint_dir / "model.safetensors").exists():
        return load_safetensors(str(checkpoint_dir / "model.safetensors"))
    for filename in ["pytorch_model.bin", "model.bin"]:
        path = checkpoint_dir / filename
        if path.exists():
            return torch.load(path, map_location="cpu")
    raise FileNotFoundError(f"No model weight file found in {checkpoint_dir}")


def clean_custom_config(config, defaults, variant):
    cfg = dict(defaults)
    cfg.update({k: v for k, v in config.items() if k in defaults})
    if "output_dim" in cfg:
        cfg["output_dim"] = num_labels(variant)
    if "num_labels" in cfg:
        cfg["num_labels"] = num_labels(variant)
    if "kernel_sizes" in cfg and isinstance(cfg["kernel_sizes"], list):
        cfg["kernel_sizes"] = tuple(cfg["kernel_sizes"])
    return cfg


def build_custom_model(spec, checkpoint_dir):
    raw_cfg = read_custom_config(checkpoint_dir)
    cfg = clean_custom_config(raw_cfg, spec["defaults"], spec["variant"])
    builder = spec["builder"]

    if builder == "cnn":
        model = MultiScaleCNN1DModel(**cfg)
    elif builder == "rnn":
        model = ImprovedRNNModel(**cfg)
    elif builder == "lstm":
        model = LSTM(**cfg)
    elif builder == "gru":
        model = BiGRUSwiGLUModel(**cfg)
    else:
        raise ValueError(f"Unknown custom builder: {builder}")

    state_dict = load_state_dict_from_dir(checkpoint_dir)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if unexpected or missing:
        print(f"[{spec['name']}] missing keys: {len(missing)}, unexpected keys: {len(unexpected)}")
    return model


def load_transformer_model(spec, checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    variant_label2idx, variant_idx2label = get_label_map(spec["variant"])
    try:
        return AutoModelForTokenClassification.from_pretrained(
            str(checkpoint_dir),
            num_labels=num_labels(spec["variant"]),
            id2label=variant_idx2label,
            label2id=variant_label2idx,
            ignore_mismatched_sizes=True,
        )
    except Exception as first_error:
        print(f"[{spec['name']}] Direct from_pretrained failed, fallback to base model + state_dict:", first_error)
        model = AutoModelForTokenClassification.from_pretrained(
            spec["base_model"],
            num_labels=num_labels(spec["variant"]),
            id2label=variant_idx2label,
            label2id=variant_label2idx,
            ignore_mismatched_sizes=True,
        )
        state_dict = load_state_dict_from_dir(checkpoint_dir)
        model.load_state_dict(state_dict, strict=False)
        return model


def load_model_for_spec(spec, checkpoint_dir):
    if spec["type"] == "transformer":
        return load_transformer_model(spec, checkpoint_dir)
    if spec["type"] == "custom":
        return build_custom_model(spec, checkpoint_dir)
    raise ValueError(f"Unknown model type: {spec['type']}")


In [ ]:
def labels_from_predictions(predictions, labels, idx2label):
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    pred_ids = np.argmax(predictions, axis=-1)

    true_labels = []
    true_predictions = []
    for pred_seq, label_seq in zip(pred_ids, labels):
        current_labels = []
        current_predictions = []
        for pred_id, label_id in zip(pred_seq, label_seq):
            if int(label_id) != IGNORE_INDEX:
                current_labels.append(idx2label[int(label_id)])
                current_predictions.append(idx2label[int(pred_id)])
        true_labels.append(current_labels)
        true_predictions.append(current_predictions)
    return true_labels, true_predictions


def round3(value):
    if value is None:
        return np.nan
    return round(float(value), 3)


def entity_name(bio_label):
    if bio_label == "O":
        return "O"
    return bio_label.split("-", 1)[-1]


def token_accuracy_by_entity(true_labels, true_predictions, entity_labels):
    correct = {label: 0 for label in entity_labels}
    total = {label: 0 for label in entity_labels}

    for labels, predictions in zip(true_labels, true_predictions):
        for true_label, pred_label in zip(labels, predictions):
            entity = entity_name(true_label)
            if entity in total:
                total[entity] += 1
                correct[entity] += int(true_label == pred_label)

    per_label = {
        label: (correct[label] / total[label] if total[label] else np.nan)
        for label in entity_labels
    }
    supports = np.array([total[label] for label in entity_labels], dtype=float)
    values = np.array([
        per_label[label] if not np.isnan(per_label[label]) else np.nan
        for label in entity_labels
    ], dtype=float)

    macro = np.nanmean(values) if len(values) else np.nan
    weighted = np.nansum(values * supports) / supports.sum() if supports.sum() else np.nan
    return per_label, macro, weighted


def build_metric_rows(model_name, model_type, report_dict, true_labels, true_predictions):
    entity_labels = [
        label for label, metrics in report_dict.items()
        if isinstance(metrics, dict) and not label.endswith("avg")
    ]

    rows = {metric: {"type": model_type, "model": model_name} for metric in ["f1", "recall", "precision", "accuracy"]}
    metric_to_report_key = {
        "f1": "f1-score",
        "recall": "recall",
        "precision": "precision",
    }

    for metric, report_key in metric_to_report_key.items():
        row = rows[metric]
        for label in entity_labels:
            row[label] = round3(report_dict[label].get(report_key))
        row["micro"] = round3(report_dict.get("micro avg", {}).get(report_key))
        row["macro"] = round3(report_dict.get("macro avg", {}).get(report_key))
        row["weighted"] = round3(report_dict.get("weighted avg", {}).get(report_key))

    per_label_acc, macro_acc, weighted_acc = token_accuracy_by_entity(
        true_labels,
        true_predictions,
        entity_labels,
    )
    acc_row = rows["accuracy"]
    for label in entity_labels:
        acc_row[label] = round3(per_label_acc[label])
    acc_row["micro"] = round3(accuracy_score(true_labels, true_predictions))
    acc_row["macro"] = round3(macro_acc)
    acc_row["weighted"] = round3(weighted_acc)

    return rows, entity_labels


def save_metric_table(metric_name, rows, label_columns):
    df = pd.DataFrame(rows)
    ordered_columns = ["type", "model"] + label_columns + ["micro", "macro", "weighted"]
    for column in ordered_columns:
        if column not in df.columns:
            df[column] = np.nan
    df = df[ordered_columns]
    df.to_csv(RESULTS_DIR / f"{metric_name}.csv", index=False)
    return df


def make_training_args(model_name, batch_size):
    return TrainingArguments(
        output_dir=str(RESULTS_DIR / "trainer_tmp" / model_name.replace("/", "_")),
        per_device_eval_batch_size=batch_size,
        report_to="none",
        dataloader_pin_memory=torch.cuda.is_available(),
    )


In [ ]:
metric_rows = {
    "f1": [],
    "recall": [],
    "precision": [],
    "accuracy": [],
}
all_entity_labels = set()
failures = []

for spec in MODEL_SPECS:
    model_name = spec["name"]
    model_type = spec["variant"]
    print()
    print("=" * 80)
    print(f"Evaluating {model_name} ({model_type})")

    try:
        checkpoint_dir = find_checkpoint_dir(spec)
        if checkpoint_dir is None:
            raise FileNotFoundError(f"No checkpoint folder found for {model_name}. Tried: {spec['folders']}")
        print("Checkpoint:", checkpoint_dir)

        tokenizer = get_tokenizer(spec["tokenizer_key"])
        _, variant_idx2label = get_label_map(model_type)
        tokenized_dataset = get_tokenized_dataset(
            variant=model_type,
            tokenizer_key=spec["tokenizer_key"],
            strategy=spec["strategy"],
            max_length=spec.get("max_length"),
        )
        test_dataset = tokenized_dataset["test"]
        data_collator = DataCollatorForTokenClassification(
            tokenizer=tokenizer,
            padding=True,
            label_pad_token_id=IGNORE_INDEX,
        )

        model = load_model_for_spec(spec, checkpoint_dir).to(device)
        trainer = Trainer(
            model=model,
            args=make_training_args(model_name, spec.get("batch_size", 64)),
            data_collator=data_collator,
        )

        prediction_output = trainer.predict(test_dataset)
        true_labels, true_predictions = labels_from_predictions(
            prediction_output.predictions,
            prediction_output.label_ids,
            variant_idx2label,
        )

        report_dict = classification_report(
            true_labels,
            true_predictions,
            output_dict=True,
            zero_division=0,
        )
        report_text = classification_report(
            true_labels,
            true_predictions,
            digits=3,
            zero_division=0,
        )
        print(report_text)

        rows, entity_labels = build_metric_rows(
            model_name=model_name,
            model_type=model_type,
            report_dict=report_dict,
            true_labels=true_labels,
            true_predictions=true_predictions,
        )
        all_entity_labels.update(entity_labels)
        for metric_name, row in rows.items():
            metric_rows[metric_name].append(row)

        del model, trainer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as exc:
        failure = {
            "type": model_type,
            "model": model_name,
            "error": repr(exc),
            "folders_tried": spec["folders"],
        }
        failures.append(failure)
        print("FAILED:", failure)

label_columns = sorted(all_entity_labels)
metric_dfs = {
    metric_name: save_metric_table(metric_name, rows, label_columns)
    for metric_name, rows in metric_rows.items()
}

if failures:
    pd.DataFrame(failures).to_csv(RESULTS_DIR / "failures.csv", index=False)

print("Saved metric files to:", RESULTS_DIR)
for metric_name, df in metric_dfs.items():
    print()
    print(f"{metric_name}.csv")
    display(df)

if failures:
    display(pd.DataFrame(failures))



Evaluating CNN (word)
Checkpoint: /content/drive/MyDrive/Mini Project NLP/CNN


Map (num_proc=2):   0%|          | 0/5027 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/3000 [00:00<?, ? examples/s]

                     precision    recall  f1-score   support

                AGE      0.934     0.954     0.944       567
               DATE      0.955     0.971     0.963      1644
             GENDER      0.921     0.935     0.928       448
                JOB      0.567     0.514     0.539       173
           LOCATION      0.849     0.879     0.863      4408
               NAME      0.898     0.809     0.851       314
       ORGANIZATION      0.728     0.742     0.735       770
         PATIENT_ID      0.955     0.962     0.959      1965
SYMPTOM_AND_DISEASE      0.793     0.745     0.769      1127
     TRANSPORTATION      0.933     0.865     0.898       193

          micro avg      0.874     0.882     0.878     11609
          macro avg      0.853     0.838     0.845     11609
       weighted avg      0.874     0.882     0.878     11609


Evaluating RNN (word)
Checkpoint: /content/drive/MyDrive/Mini Project NLP/RNN


...

,type,model,AGE,DATE,GENDER,JOB,LOCATION,NAME,ORGANIZATION,PATIENT_ID,SYMPTOM_AND_DISEASE,TRANSPORTATION,micro,macro,weighted
0,word,CNN,0.954,0.971,0.935,0.514,0.879,0.809,0.742,0.962,0.745,0.865,0.882,0.838,0.882
1,word,RNN,0.950,0.960,0.926,0.451,0.848,0.745,0.735,0.952,0.721,0.839,0.861,0.813,0.861
2,word,LSTM,0.971,0.960,0.950,0.462,0.870,0.733,0.755,0.970,0.746,0.819,0.878,0.824,0.878
3,word,GRU,0.954,0.966,0.946,0.439,0.870,0.783,0.732,0.962,0.737,0.824,0.875,0.821,0.875
4,word,PhoBERT,0.974,0.993,0.981,0.757,0.958,0.928,0.894,0.986,0.879,0.974,0.954,0.932,0.954
5,word,ViDeBERTa,0.973,0.993,0.968,0.711,0.943,0.918,0.878,0.986,0.861,0.969,0.944,0.920,0.944
6,syllable,DistilBERT,0.969,0.990,0.950,0.543,0.930,0.893,0.850,0.979,0.794,0.943,0.925,0.884,0.925
7,syllable,ELECTRA,0.976,0.992,0.978,0.786,0.952,0.969,0.898,0.980,0.864,0.974,0.951,0.937,0.951
8,syllable,XLM-RoBERTa,0.973,0.992,0.978,0.769,0.952,0.962,0.886,0.986,0.853,0.984,0.949,0.933,0.949



precision.csv


,type,model,AGE,DATE,GENDER,JOB,LOCATION,NAME,ORGANIZATION,PATIENT_ID,SYMPTOM_AND_DISEASE,TRANSPORTATION,micro,macro,weighted
0,word,CNN,0.934,0.955,0.921,0.567,0.849,0.898,0.728,0.955,0.793,0.933,0.874,0.853,0.874
1,word,RNN,0.939,0.941,0.932,0.605,0.832,0.919,0.711,0.954,0.791,0.895,0.866,0.852,0.865
2,word,LSTM,0.922,0.962,0.926,0.567,0.868,0.907,0.766,0.946,0.811,0.849,0.885,0.852,0.884
3,word,GRU,0.930,0.941,0.926,0.555,0.836,0.877,0.698,0.942,0.771,0.878,0.861,0.835,0.859
4,word,PhoBERT,0.964,0.981,0.964,0.716,0.946,0.881,0.877,0.981,0.887,0.984,0.943,0.918,0.944
5,word,ViDeBERTa,0.956,0.979,0.965,0.737,0.928,0.904,0.852,0.977,0.865,0.959,0.932,0.912,0.932
6,syllable,DistilBERT,0.935,0.979,0.926,0.514,0.905,0.871,0.791,0.972,0.800,0.915,0.904,0.861,0.905
7,syllable,ELECTRA,0.963,0.987,0.962,0.720,0.943,0.895,0.844,0.979,0.864,0.935,0.937,0.909,0.938
8,syllable,XLM-RoBERTa,0.966,0.984,0.954,0.696,0.939,0.942,0.859,0.974,0.852,0.955,0.936,0.912,0.936



accuracy.csv


,type,model,AGE,DATE,GENDER,JOB,LOCATION,NAME,ORGANIZATION,PATIENT_ID,SYMPTOM_AND_DISEASE,TRANSPORTATION,micro,macro,weighted
0,word,CNN,0.948,0.986,0.935,0.456,0.895,0.804,0.814,0.950,0.764,0.802,0.962,0.835,0.882
1,word,RNN,0.944,0.977,0.926,0.387,0.865,0.743,0.807,0.940,0.727,0.790,0.956,0.811,0.860
2,word,LSTM,0.964,0.980,0.950,0.422,0.883,0.734,0.822,0.960,0.754,0.763,0.961,0.823,0.876
3,word,GRU,0.952,0.980,0.946,0.394,0.885,0.779,0.815,0.952,0.756,0.790,0.960,0.825,0.876
4,word,PhoBERT,0.974,0.997,0.981,0.735,0.965,0.927,0.916,0.973,0.891,0.966,0.983,0.933,0.953
5,word,ViDeBERTa,0.973,0.995,0.968,0.707,0.943,0.918,0.910,0.974,0.873,0.927,0.979,0.919,0.939
6,syllable,DistilBERT,0.969,0.992,0.948,0.600,0.951,0.841,0.892,0.966,0.804,0.898,0.971,0.886,0.925
7,syllable,ELECTRA,0.976,0.994,0.976,0.819,0.962,0.955,0.923,0.968,0.871,0.958,0.979,0.940,0.948
8,syllable,XLM-RoBERTa,0.973,0.994,0.978,0.796,0.966,0.933,0.914,0.976,0.867,0.977,0.980,0.938,0.949
